# TEM nanoparticle polydispersity

Automatically measures the diameter of spherical nanoparticles in Bright Field TEM images (packed particles, possibly touching each other) and computes polydispersity statistics (mean, standard deviation, CV%, PDI).

**Pipeline**
1. Pixel -> nm calibration via the black scale bar present in the image.
2. Thresholding (Otsu) to separate particles (dark) from background (light).
3. Watershed segmentation (based on the distance transform) to separate particles that are touching each other.
4. Measurement of area/equivalent diameter/circularity for each particle, filtering out fragments that are too small or not circular enough (noise).
5. Computation of polydispersity statistics + histogram + verification overlay.

**Dependencies**: `opencv-python-headless`, `numpy`, `scipy`, `scikit-image`, `matplotlib` (see `requirements.txt`).

## Configuration

Set the path to your TEM image (and, if needed, the other parameters) here before running the rest of the notebook.

In [ ]:
image_path = "image.jpg"  # <-- set the path to your TEM image
scalebar_nm = 200
scalebar_region = (0.85, 1.0, 0.0, 0.35)
outdir = "results_TEM"

In [ ]:
import os

import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Scale bar calibration

In [ ]:
def calibrate_scalebar(gray_img, scalebar_nm=200.0, region=(0.85, 1.0, 0.0, 0.35),
                        dark_thresh=100, min_bar_px=30):
    """
    Finds the length in pixels of the scale bar (solid, horizontal black
    bar) within a sub-region of the image and computes the nm/pixel
    conversion factor.

    Parameters
    ----------
    gray_img : np.ndarray (2D, uint8)
        Grayscale image.
    scalebar_nm : float
        Real length of the scale bar in nm (e.g. 200).
    region : tuple(float, float, float, float)
        Fractions (row_start, row_end, col_start, col_end) of the image
        region in which to search for the scale bar. Default: bottom-left
        corner.
    dark_thresh : int
        Intensity threshold below which a pixel is considered "black"
        (part of the scale bar).
    min_bar_px : int
        Minimum number of consecutive dark pixels in a row for it to be
        considered a scale bar row (avoids confusing the "200 nm" text
        with the bar itself).

    Returns
    -------
    dict with:
        nm_per_px : float
        bar_length_px : float
        bar_bbox : (row, col_start, col_end) in the original image coordinates
    """
    h, w = gray_img.shape
    r0, r1, c0, c1 = region
    r0, r1 = int(h * r0), int(h * r1)
    c0, c1 = int(w * c0), int(w * c1)
    crop = gray_img[r0:r1, c0:c1]

    dark_mask = crop < dark_thresh

    best_row, best_start, best_end, best_len = None, None, None, 0
    for row in range(dark_mask.shape[0]):
        cols = np.where(dark_mask[row])[0]
        if len(cols) < min_bar_px:
            continue
        # the bar is a continuous segment: take min/max and check that
        # most pixels in that range are actually dark
        # (to exclude text rows, which are more "sparse")
        span = cols.max() - cols.min() + 1
        if span < min_bar_px:
            continue
        fill_ratio = len(cols) / span
        if fill_ratio > 0.9 and span > best_len:
            best_row, best_start, best_end, best_len = row, cols.min(), cols.max(), span

    if best_row is None:
        raise RuntimeError(
            "Scale bar not found automatically: try changing 'region' "
            "or 'dark_thresh'."
        )

    bar_length_px = best_end - best_start + 1
    nm_per_px = scalebar_nm / bar_length_px

    return {
        "nm_per_px": nm_per_px,
        "bar_length_px": bar_length_px,
        "bar_bbox": (best_row + r0, best_start + c0, best_end + c0),
    }

## 2-3. Segmentation (Otsu) + splitting touching particles (watershed)

In [ ]:
def segment_and_split_particles(gray_img, exclude_region=None, min_peak_distance=15):
    """
    Segments dark particles on a light background and splits touching
    particles using watershed based on the distance transform.

    Parameters
    ----------
    gray_img : np.ndarray (2D, uint8)
    exclude_region : tuple(float, float, float, float), optional
        Region (row_start, row_end, col_start, col_end), in fractions, to
        exclude from segmentation (e.g. where the scale bar/text is
        located), set to background color before thresholding.
    min_peak_distance : int
        Minimum distance (in pixels) between local maxima of the distance
        transform, used as markers for the watershed. Should be chosen
        based on the approximate particle radius (smaller if particles are
        small/closely packed).

    Returns
    -------
    labels_ws : np.ndarray (2D, int) - watershed labels (0 = background)
    mask : np.ndarray (2D, bool) - binary mask of the particles
    """
    h, w = gray_img.shape
    work = gray_img.copy()

    if exclude_region is not None:
        r0, r1, c0, c1 = exclude_region
        r0, r1 = int(h * r0), int(h * r1)
        c0, c1 = int(w * c0), int(w * c1)
        work[r0:r1, c0:c1] = 255  # force to "light background"

    blur = cv2.GaussianBlur(work, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = np.ones((3, 3), np.uint8)
    th_clean = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)
    th_clean = cv2.morphologyEx(th_clean, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = th_clean > 0

    dist = ndi.distance_transform_edt(mask)
    coords = peak_local_max(dist, min_distance=min_peak_distance, labels=mask,
                             exclude_border=False)
    markers_mask = np.zeros(dist.shape, dtype=bool)
    markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)

    labels_ws = watershed(-dist, markers, mask=mask)
    return labels_ws, mask

## 4. Particle measurement + noise filtering

In [ ]:
def measure_particles(labels_ws, nm_per_px, min_diam_px=25, min_circularity=0.5):
    """
    Computes area, equivalent diameter and circularity for each watershed
    region and filters out regions that are too small or not circular
    enough (typically noise fragments or edge artifacts).

    Returns
    -------
    list of dict, one per accepted particle, with keys:
        label, area_px, eq_diam_px, eq_diam_nm, circularity, solidity, centroid
    """
    props = regionprops(labels_ws)
    particles = []
    for p in props:
        area = p.area
        perim = p.perimeter if p.perimeter > 0 else 1
        circularity = 4 * np.pi * area / (perim ** 2)
        eq_diam_px = 2 * np.sqrt(area / np.pi)

        if eq_diam_px < min_diam_px:
            continue
        if circularity < min_circularity:
            continue

        particles.append({
            "label": p.label,
            "area_px": area,
            "eq_diam_px": eq_diam_px,
            "eq_diam_nm": eq_diam_px * nm_per_px,
            "circularity": circularity,
            "solidity": p.solidity,
            "centroid": p.centroid,  # (row, col)
        })
    return particles

## Polydispersity statistics

In [ ]:
def polydispersity_stats(diam_nm):
    """
    Computes polydispersity statistics from an array of diameters (in nm).

    PDI here is defined as (sigma/mean)^2, the same convention typically
    used in DLS. It is a dimensionless metric: lower values indicate a
    more monodisperse sample (e.g. PDI < 0.05 - 0.1 in TEM).
    """
    diam_nm = np.asarray(diam_nm)
    mean_d = diam_nm.mean()
    std_d = diam_nm.std(ddof=1)
    return {
        "n": len(diam_nm),
        "mean_nm": mean_d,
        "median_nm": float(np.median(diam_nm)),
        "std_nm": std_d,
        "cv_percent": std_d / mean_d * 100,
        "pdi": (std_d / mean_d) ** 2,
        "min_nm": float(diam_nm.min()),
        "max_nm": float(diam_nm.max()),
    }

## Visualization

In [ ]:
def draw_overlay(color_img, particles, out_path=None):
    overlay = color_img.copy()
    for p in particles:
        cy, cx = p["centroid"]
        r = p["eq_diam_px"] / 2
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), int(round(r)),
                   (0, 0, 255), 2)
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), 1, (0, 255, 0), 2)
    if out_path:
        cv2.imwrite(out_path, overlay)
    return overlay


def show_overlay(overlay_bgr, n_particles=None):
    """Displays a BGR (OpenCV) overlay image inline using matplotlib."""
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB))
    title = "Detected particles"
    if n_particles is not None:
        title += f" (N={n_particles})"
    plt.title(title)
    plt.axis("off")
    plt.show()


def plot_histogram(diam_nm, stats, out_path=None, material="nanoparticles"):
    plt.figure(figsize=(7, 5))
    plt.hist(diam_nm, bins=30, color="#4C72B0", edgecolor="white")
    plt.axvline(stats["mean_nm"], color="crimson", linestyle="--", linewidth=1.5,
                label=f'Mean = {stats["mean_nm"]:.2f} nm')
    plt.xlabel("Particle diameter (nm)")
    plt.ylabel("Count")
    plt.title(
        f'Size distribution {material} (N={stats["n"]})\n'
        f'Std dev = {stats["std_nm"]:.2f} nm, CV = {stats["cv_percent"]:.1f}%, '
        f'PDI = {stats["pdi"]:.4f}'
    )
    plt.legend()
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=150)
    plt.show()

## Full pipeline

In [ ]:
def analyze_image(image_path, scalebar_nm=200.0,
                   scalebar_region=(0.85, 1.0, 0.0, 0.35),
                   min_peak_distance=15, min_diam_px=25, min_circularity=0.5,
                   material="nanoparticles", outdir=None):
    """
    Runs the entire pipeline on a TEM image and returns a dictionary with
    calibration, particle list, statistics and the overlay image (BGR). If
    outdir is specified, also saves overlay.png and histogram.png.
    """
    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"Could not open image: {image_path}")

    calib = calibrate_scalebar(gray, scalebar_nm=scalebar_nm, region=scalebar_region)

    labels_ws, mask = segment_and_split_particles(
        gray, exclude_region=scalebar_region, min_peak_distance=min_peak_distance
    )

    particles = measure_particles(
        labels_ws, nm_per_px=calib["nm_per_px"],
        min_diam_px=min_diam_px, min_circularity=min_circularity,
    )
    diam_nm = np.array([p["eq_diam_nm"] for p in particles])
    stats = polydispersity_stats(diam_nm)

    color_img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    overlay_path = os.path.join(outdir, "overlay.png") if outdir else None
    overlay = draw_overlay(color_img, particles, overlay_path)

    result = {"calibration": calib, "particles": particles, "stats": stats, "overlay": overlay}

    if outdir:
        os.makedirs(outdir, exist_ok=True)
        plot_histogram(diam_nm, stats, os.path.join(outdir, "histogram.png"),
                        material=material)

    return result

## Run pipeline

Uses the `image_path` and parameters set in the **Configuration** cell above. Displays both the size-distribution histogram and the detected-particles overlay (particles as red circles, centroid as green dot) inline.

In [ ]:
result = analyze_image(
    image_path,
    scalebar_nm=scalebar_nm,
    scalebar_region=scalebar_region,
    outdir=outdir,
)

show_overlay(result["overlay"], n_particles=result["stats"]["n"])

result["stats"]